# Demo: Unlearning on Synthetic Time Series (Energy)

This notebook generates a synthetic dataset, saves it to CSV, loads it via the dataset-agnostic DataManager, and runs the exact-retraining unlearning baseline.



In [ ]:
import os
import numpy as np
import pandas as pd

from machine_unlearning_tool.schemas import DatasetSchema
from machine_unlearning_tool.loaders import CsvAdapter
from machine_unlearning_tool.data_manager import DataManager
from machine_unlearning_tool.data_utils import extract_features_targets
from machine_unlearning_tool.workflow import run_exact_retraining


In [ ]:
# 1) Generate synthetic dataset and save to CSV
base_dir = "/Users/jeffkroon/localdocuments/Thesis"
data_dir = os.path.join(base_dir, "data")
os.makedirs(data_dir, exist_ok=True)

n = 1200
rng = np.random.default_rng(42)
ids = np.arange(n)
client_ids = rng.choice(["A", "B", "C"], size=n, replace=True)

# Synthetic features
feat1 = rng.normal(0.0, 1.0, size=n).astype(np.float32)
feat2 = rng.normal(0.0, 1.0, size=n).astype(np.float32)
feat3 = rng.normal(0.0, 1.0, size=n).astype(np.float32)

# Create a target with temporal dependency (simple AR-like behavior)
noise = rng.normal(0.0, 0.3, size=n).astype(np.float32)
target = np.zeros(n, dtype=np.float32)
for t in range(n):
    lag = target[t-1] if t > 0 else 0.0
    target[t] = 0.6 * lag + 0.4 * (0.5*feat1[t] - 0.2*feat2[t] + 0.1*feat3[t]) + noise[t]

# Assemble DataFrame (ordered by id -> acts as time index)
df = pd.DataFrame({
    "id": ids,
    "client_id": client_ids,
    "feat1": feat1,
    "feat2": feat2,
    "feat3": feat3,
    "target": target,
})

csv_path = os.path.join(data_dir, "demo_energy.csv")
df.to_csv(csv_path, index=False)
print(f"Saved synthetic CSV to: {csv_path}")


In [ ]:
# 2) Define schema and create DataManager
schema = DatasetSchema(
    id_column="id",
    timestamp_column=None,
    input_cols=["feat1", "feat2", "feat3"],
    target_col="target",
    client_column="client_id",
    rename_map=None,
)

adapter = CsvAdapter()
dm = DataManager(adapter=adapter, schema=schema, seq_len=24, batch_size=64, seed=42)

loaded_df = dm.load_dataframe(csv_path)
print(loaded_df.head())


In [ ]:
# 3) Create loaders (train/val/test) with optional forget_ids
forget_ids = list(range(50, 80))  # demo: forget a small window
loaders = dm.prepare_loaders(
    loaded_df,
    forget_ids=forget_ids,
    train_frac=0.7,
    val_frac=0.15,
    scale_inputs=True,
)

for k, v in loaders.items():
    print(k, len(v.dataset))


In [ ]:
# 4) Run exact-retraining baseline via workflow API
X, y = extract_features_targets(loaded_df, schema.input_cols, schema.target_col)
res = run_exact_retraining(
    X=X,
    y=y,
    df=loaded_df,
    input_cols=schema.input_cols,
    target_col=schema.target_col,
    id_column=schema.id_column,
    forget_ids=forget_ids,
    device=None,  # auto CUDA if available
    seq_len=24,
    model_params={"hidden_size": 64},
    train_params={"epochs": 3, "batch_size": 64},
)

print("Retain metrics:", res["metrics_retain"])
